# GPU Programming and Performance
This notebook introduces **GPU computing in Python**, explaining the differences between CPU, GPU, and TPU architectures. It covers data parallelism, the GPU execution hierarchy, memory management, and performance profiling, with hands-on exercises and CuPy to compare CPU and GPU computations. Each section includes clear explanations and examples.

# Part 1: Conceptual Introduction

### From CPU Parallelism to GPU Acceleration

In the previous week we explored several techniques that improve program performance on CPU-based systems:

- threading
- multiprocessing
- asynchronous programming

These approaches allow programs to run tasks concurrently, but they still operate within the limits of **CPU architecture**.

Most CPUs contain only a small number of cores (typically between 4 and 16). For workloads that involve **large numerical computations**, a different architecture becomes more effective.

Graphics Processing Units (GPUs) are designed to process **thousands of operations simultaneously**.

This architecture makes GPUs particularly useful for:

- machine learning
- scientific simulations
- image processing
- large-scale matrix operations

### CPU vs GPU Computation Models

| Feature | CPU | GPU |
|------|------|------|
| Number of cores | few powerful cores (8–16 powerful cores) | thousands of lightweight cores |
| Optimisation | sequential tasks | parallel numerical workloads |
| Typical use | operating systems, applications | AI and numerical computing |


### Data Parallelism

Most GPU workloads rely on **data parallelism**. The same operation is applied
to many independent pieces of data simultaneously.

#### Example: vector addition

```
c[i] = a[i] + b[i]
```

Each element can be computed independently.

A GPU assigns **one thread to each element**, allowing thousands of operations to run simultaneously.

Data parallelism appears frequently in:

- neural networks
- matrix operations
- scientific computing


### GPU Execution Model

GPU programs follow a hierarchical execution structure.

```
GPU Execution Hierarchy

Grid
 └── Blocks
      └── Threads
```

Definitions:

| Concept | Description |
|------|------|
| Thread | smallest execution unit |
| Block | group of threads |
| Grid | collection of blocks executing a kernel |

Thousands of threads run simultaneously across blocks.


### CUDA Conceptual Overview

Most GPUs are programmed using **Compute Unified Device Architecture (CUDA)**.

CUDA introduces the concept of a **kernel**.

Definition:

```
A kernel is a function executed
by thousands of GPU threads
in parallel.
```

Execution model:

```
CPU launches kernel
GPU executes kernel
each thread processes part of the data
```

### Python Libraries for GPU Computing

Instead of writing CUDA C code, Python can access GPUs using libraries.

Common options:

| Library | Purpose |
|------|------|
|CuPy| NumPy compatible GPU arrays|
|PyTorch| Deep learning and tensor computation|
|TensorFlow| Machine learning frameworks|

For this practical lecture, we focus on  **CuPy** mirroring **NumPy**  syntax.

---
# Part 2: Environment Setup
### Setting up Google Colab
You may use **Google Colab** as it provides free GPU access.

#### Enable GPU Runtime

Navigate to:

```
Runtime → Change runtime type
```

Select:

```
Runtime type → Python
Hardware accelerator → T4 GPU
```

### Verify GPU Availability

Run:
```python
!nvidia-smi
```

The output shows:

- GPU model
- GPU memory
- CUDA driver information

In [ ]:
!nvidia-smi

### Installing Required Libraries
> Install CuPy:

```python
!pip install cupy-cuda12x
```

Import required libraries:

```python
import numpy as np
import cupy as cp
import time
```

In [ ]:
!pip install cupy-cuda12x

In [ ]:
import numpy as np
import cupy as cp
import time

# Part 3: First GPU Computations
### CPU Numerical Computation
This code measures the time required to perform element-wise addition of two large NumPy arrays.
Example using NumPy:

```python
size = 10_000_000

a = np.random.rand(size)
b = np.random.rand(size)

start = time.perf_counter()
c = a + b
end = time.perf_counter()

print("CPU time:", end - start)
```

Computation occurs entirely on the CPU.


In [ ]:
# CPU Array Addition Benchmark
import numpy as np
import time

# Define array size
size = 10_000_000  # 10 million elements

# Create two random arrays of the given size
a = np.random.rand(size) # generates 10 million random floats uniformly distributed in the range [0.0, 1.0)
b = np.random.rand(size)

# Measure CPU computation time
start = time.perf_counter()  # start timer
c = a + b                    # perform element-wise addition
end = time.perf_counter()    # end timer

# Display elapsed time
print("CPU time:", end - start)

### GPU Numerical Computation

This code measures the time required to perform element-wise addition of two large arrays on the GPU.
Equivalent computation using CuPy:

```python
size = 10_000_000

a = cp.random.rand(size)
b = cp.random.rand(size)

start = time.perf_counter()
c = a + b

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("GPU time:", end - start)
```


Important concept:

GPU operations are **asynchronous**, therefore synchronisation is required before measuring runtime.


In [ ]:
import cupy as cp       # CuPy is a GPU array library similar to NumPy
import time             # For measuring execution time

""""
For fair GPU benchmarking, it is good practice to perform a warm-up before measuring. 
The first GPU operation often includes one-time overheads, such as kernel initialisation
"""
# Warm-up
_ = a + b
cp.cuda.Stream.null.synchronize()

# Define the size of the arrays
size = 10_000_000       # 10 million elements

# Generate two random arrays directly on the GPU
a = cp.random.rand(size)  # Random floats in [0.0, 1.0)
b = cp.random.rand(size)  # Random floats in [0.0, 1.0)

# Start high-resolution timer
start = time.perf_counter()

# Perform element-wise addition on the GPU
c = a + b

# Synchronize GPU to ensure all operations are complete before stopping the timer
# CuPy operations are asynchronous by default, so without this, timing would be inaccurate
cp.cuda.Stream.null.synchronize()

# Stop the timer
end = time.perf_counter()

# Display elapsed GPU computation time
print("GPU time:", end - start)

### CPU vs GPU Timing Benchmark (NumPy vs CuPy)

This code compares the execution time of **matrix multiplication** on **CPU (using NumPy)** and **GPU (using CuPy)** for different array sizes. It measures the time for matrix multiplication for small, medium, and large arrays.


In [ ]:
import numpy as np        # NumPy for CPU computations
import cupy as cp         # CuPy for GPU computations
import time               # For measuring execution time

# Function to measure CPU computation time for matrix multiplication
def cpu_time(array_size):
    # Generate a square matrix of shape (array_size, array_size) with random floats in [0.0, 1.0)
    x_cpu = np.random.rand(array_size, array_size)

    # Start timer
    start = time.time()

    # Perform matrix multiplication (dot product) on CPU
    y_cpu = np.dot(x_cpu, x_cpu)

    # Stop timer
    end = time.time()

    # Return elapsed CPU time
    return end - start

# Function to measure GPU computation time for matrix multiplication
def gpu_time(array_size):
    # Generate a square matrix of shape (array_size, array_size) directly on GPU memory
    x_gpu = cp.random.rand(array_size, array_size)

    # Start timer
    start = time.time()

    # Perform matrix multiplication (dot product) on GPU
    y_gpu = cp.dot(x_gpu, x_gpu)

    # Synchronize GPU to ensure computation completes before stopping the timer
    # CuPy operations are asynchronous by default
    cp.cuda.Stream.null.synchronize()

    # Stop timer
    end = time.time()

    # Return elapsed GPU time
    return end - start

# Define array sizes for benchmarking: small, medium, large
sizes = [1000, 5000, 10000]

print("CPU vs GPU Timing Benchmark:")

# Loop over each array size and measure CPU and GPU times
for size in sizes:
    t_cpu = cpu_time(size)
    t_gpu = gpu_time(size)

    print(f"Array size: {size}×{size}")
    print(f"  CPU time: {t_cpu:.4f} s")
    print(f"  GPU time: {t_gpu:.4f} s\n")

# Part 4: Memory and Performance

### GPU Memory Management

GPUs have their own memory (device memory), which is separate from the CPU’s memory (host memory). This distinction is crucial because data must be explicitly transferred between CPU and GPU memory before computation and brought back after computation.

| Memory Type | Description |
|------|------|
| Host memory | Regular RAM used by the CPU. This is where your standard NumPy arrays reside |
| Device memory | GPU RAM where computations happen when using libraries like CuPy or other relevant libraries. |

**Note**:

Data transfers between CPU and GPU can affect performance.


All of the data we have worked with so far has been created using **NumPy on the CPU**. In order for `CuPy` to operate on this data on the GPU, it has been **implicitly copying data back and forth**. This data movement incurs a **performance penalty**, which can be significant for large arrays.

We also have the option of **taking explicit control of the data transfer**. By using `cp.asarray` with `copy=True`, we can move NumPy arrays to the GPU **in advance**, ensuring that computations occur entirely on GPU memory and avoiding repeated implicit transfers.

The following example demonstrates the difference between **implicit transfer** and **explicit transfer** with timing measurements:


In [ ]:
import numpy as np
import cupy as cp
import time

# --------------------------
# Create array in CPU memory (host)
# --------------------------
host_array = np.arange(10_000_000)
print("Created array in CPU memory (host)")

# --------------------------
# Implicit transfer via cp.array()
# --------------------------
start_implicit = time.time()
gpu_array_implicit = cp.array(host_array)  # CuPy automatically transfers data
gpu_result_implicit = gpu_array_implicit * 2
cp.cuda.Stream.null.synchronize()           # Ensure GPU computation finishes
result_back_implicit = cp.asnumpy(gpu_result_implicit)
end_implicit = time.time()
print(f"Implicit transfer total time: {end_implicit - start_implicit:.5f} seconds")

# --------------------------
# Explicit transfer using cp.asarray(copy=True)
# --------------------------
start_explicit = time.time()
gpu_array_explicit = cp.asarray(host_array, copy=True)  # Explicitly transfer to GPU
gpu_result_explicit = gpu_array_explicit * 2
cp.cuda.Stream.null.synchronize()
result_back_explicit = cp.asnumpy(gpu_result_explicit)
end_explicit = time.time()
print(f"Explicit transfer total time: {end_explicit - start_explicit:.5f} seconds")

# --------------------------
# Verify correctness
# --------------------------
print("First 10 elements of result (implicit):", result_back_implicit[:10])
print("First 10 elements of result (explicit):", result_back_explicit[:10])


### Main Observations
1. **Implicit transfer (`cp.array`)**
   - Convenient, but the library decides **when the transfer occurs**.
   - Can lead to repeated or hidden data movement, affecting performance.

2. **Explicit transfer (`cp.asarray(copy=True)`)**
   - The user decides **exactly when the data is moved**.
   - Reduces overhead for **repeated GPU computations**, as data is already on the GPU.

3. **Performance impact**
   - Timing measurements show how explicit control can optimise GPU computation, especially for **large arrays** or **multiple repeated operations**.

### Performance Profiling Basics (Matrix Multiplication Experiment)
Matrix multiplication is a common workload in machine learning. Large matrix computations benefit significantly from GPU acceleration.


Performance profiling measures how long computations take.

**Example**:

```python
start = time.perf_counter()

# computation

end = time.perf_counter()

print(end - start)
```
Professional GPU profiling tools include:

| Tool | Purpose |
|------|------|
| NVIDIA Nsight | GPU profiling |
| nvprof | CUDA performance analysis |

For this course **simple runtime measurements** are sufficient.

**CPU version:**


In [ ]:
size = 2000

A = np.random.rand(size, size)
B = np.random.rand(size, size)

start = time.perf_counter()
C = A @ B # np.dot(A,B)
end = time.perf_counter()

print("CPU matrix time:", end - start)


**GPU version:**


In [ ]:
size = 2000

A = cp.random.rand(size, size)
B = cp.random.rand(size, size)

start = time.perf_counter()
C = A @ B

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("GPU matrix time:", end - start)

# Part 5: Hardware Ecosystem

### Introduction to TPUs

A **Tensor Processing Unit (TPU)** is specialised hardware designed for machine learning workloads.

| Hardware | Typical usage |
|------|------|
| CPU | general computing |
| GPU | parallel numerical computing |
| TPU | tensor operations in AI |

In Google Colab the runtime can be changed to TPU:

```
Runtime → Change runtime type → TPU
```

TPUs require specialised software libraries and cannot run standard GPU code directly.


### Now we are done with week 3's lab content 🎉🎉🎉 ##
![Week 3](cuda.png)
### Let's navigate to Exercise's notebook
[Exercise Notebook](Week3_Exercise.ipynb)